In [39]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl
import prusti_analysis as pa

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_width_chars(300)
pl.Config.set_tbl_rows(50)

polars.config.Config

In [40]:
DB_PATH = sorted(Path(".").glob("prusti-*.db"))[-1]
print(f"Loading {DB_PATH}")

df = pa.transform(pa.load_dbs([DB_PATH]))
print(f"{len(df)} rows loaded")
print(df["success"].value_counts())

Loading prusti-20260309-165527-9eba9fcdc.db
2249 rows loaded
shape: (3, 2)
┌─────────┬───────┐
│ success ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u64   │
╞═════════╪═══════╡
│ fail    ┆ 1565  │
│ timeout ┆ 32    │
│ success ┆ 652   │
└─────────┴───────┘


In [42]:
df.filter(pl.col("success") == "fail")["category"].value_counts().sort("count", descending=True)

category,count
str,u64
"""unsupported: constant scalars that are not primitives""",413
"""unsupported: trait objects""",173
"""bug: lifetime-annotated structs""",126
"""unsupported: function shapes containing alias types (pcg)""",121
"""called `Result::unwrap()` on an `Err` value: AlreadyEncoded""",120
"""unsupported: unsizing of other types than refs to arrays""",93
"""bug: const ptr offset overflow""",93
"""unsupported: invalid uninitialized bytes in constants""",71
"""unsupported: constant pointer-to-pointers""",56


In [ ]:
CATEGORY = 'expected primitive (was TyData'
df.filter((pl.col("category").str.starts_with(CATEGORY)))["file_name"]

In [ ]:
row = df.filter(pl.col("category").str.starts_with(CATEGORY)).row(1, named=True)
print(row['output'])
row

In [ ]:
print(df.group_by("test_file").agg(
    pl.len().alias("total"),
    pl.col("success").value_counts().alias("success_counts"),
    pl.col("success").value_counts(normalize=True).alias("success_ratios"),
).sort("total", descending=True))